# Day 1 — Why Polars exists: where pandas hurts at scale

*100 Days of Polars · Phase 1: Foundations and mental model*

- **Post:** [Day 1 — Why Polars Exists: Where pandas Hurts at Scale](https://medium.com/double-espresso-data/day-1-why-polars-exists-where-pandas-hurts-at-scale-63b72f6e5002)
- **Dataset:** [Olist Brazilian e-commerce](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) — CSVs in `data/raw/`
- **Written for:** Polars 2.x (run from the repo root)

In [23]:
import polars as pl

print("polars", pl.__version__)

polars 2.0.0


Same work, two libraries, real multi-table data. We measure three things:

1. Loading all 9 Olist CSVs (time and memory)
2. A realistic 4-table query (monthly revenue per category)
3. The same query in lazy mode

> Run every timing cell **twice** and keep the second number — the first run pays for disk caching.

In [24]:
import time
from pathlib import Path

import pandas as pd

DATA = Path("data/raw")

print("pandas", pd.__version__)
print("Polars thread pool:", pl.thread_pool_size(), "threads")
sorted(p.name for p in DATA.glob("*.csv"))

pandas 3.0.6
Polars thread pool: 12 threads


['olist_customers_dataset.csv',
 'olist_geolocation_dataset.csv',
 'olist_order_items_dataset.csv',
 'olist_order_payments_dataset.csv',
 'olist_order_reviews_dataset.csv',
 'olist_orders_dataset.csv',
 'olist_products_dataset.csv',
 'olist_sellers_dataset.csv',
 'product_category_name_translation.csv']

In [25]:
class timer:
    """`with timer("label") as t:` prints elapsed seconds and keeps them in t.seconds."""

    def __init__(self, label):
        self.label = label

    def __enter__(self):
        self.t0 = time.perf_counter()
        return self

    def __exit__(self, *exc):
        self.seconds = time.perf_counter() - self.t0
        print(f"{self.label:<30} {self.seconds:6.2f} s")

## 1. Load all 9 CSVs

About 1.6 million rows in total, most of them in the geolocation table. Default settings in both libraries.

In [26]:
files = sorted(DATA.glob("*.csv"))

with timer("pandas: read 9 CSVs") as t_pd:
    pd_frames = {f.stem: pd.read_csv(f) for f in files}

with timer("polars: read 9 CSVs") as t_pl:
    pl_frames = {f.stem: pl.read_csv(f) for f in files}

print(f"speed-up: {t_pd.seconds / t_pl.seconds:.1f}x")

pandas: read 9 CSVs              1.64 s
polars: read 9 CSVs              0.18 s
speed-up: 9.1x


In [27]:
# pandas needs deep=True to count string data honestly; both sides in MB (1e6 bytes)
memory = (
    pl.DataFrame(
        {
            "table": list(pd_frames),
            "rows": [len(df) for df in pd_frames.values()],
            "pandas_mb": [df.memory_usage(deep=True).sum() / 1e6 for df in pd_frames.values()],
            "polars_mb": [pl_frames[name].estimated_size() / 1e6 for name in pd_frames],
        }
    )
    .with_columns(ratio=pl.col("pandas_mb") / pl.col("polars_mb"))
    .sort("pandas_mb", descending=True)
)
memory

table,rows,pandas_mb,polars_mb,ratio
str,i64,f64,f64,f64
"""olist_geolocation_dataset""",1000163,52.557181,36.554441,1.437778
"""olist_orders_dataset""",99441,23.012839,16.61119,1.385382
"""olist_order_items_dataset""",112650,19.263282,15.65835,1.230224
"""olist_order_reviews_dataset""",99224,18.709117,13.921427,1.343908
"""olist_customers_dataset""",99441,11.569542,8.387298,1.379412
"""olist_order_payments_dataset""",103886,8.499121,6.836813,1.243141
"""olist_products_dataset""",32951,3.914933,3.403096,1.150403
"""olist_sellers_dataset""",3095,0.235872,0.16146,1.46087
"""product_category_name_translat…",71,0.003614,0.002346,1.540494


> **pandas 3 note:** pandas 3.0 stores text in an Arrow-backed `str` dtype by default (when `pyarrow` is installed), so the memory gap is smaller than the pandas 1.x/2.x days of `object` columns. Day 2 compares all three layouts.

## 2. A realistic query

**Monthly revenue per product category (English names) for delivered orders.**
`orders` → `order_items` → `products` → `category translation`: three joins, a filter, a date truncation, a group-by and a sort.

The translation CSV starts with a hidden UTF-8 byte-order mark (BOM). Both pandas and Polars strip it on read, so the header names work as-is (Day 5 shows how to check).

In [28]:
def pandas_query():
    orders = pd.read_csv(DATA / "olist_orders_dataset.csv", parse_dates=["order_purchase_timestamp"])
    items = pd.read_csv(DATA / "olist_order_items_dataset.csv")
    products = pd.read_csv(DATA / "olist_products_dataset.csv")
    trans = pd.read_csv(DATA / "product_category_name_translation.csv")

    delivered = orders.loc[orders["order_status"] == "delivered", ["order_id", "order_purchase_timestamp"]]
    df = (
        items.merge(delivered, on="order_id")
        .merge(products[["product_id", "product_category_name"]], on="product_id", how="left")
        .merge(trans, on="product_category_name", how="left")
    )
    df["month"] = df["order_purchase_timestamp"].dt.to_period("M").dt.to_timestamp()
    return (
        df.groupby(["product_category_name_english", "month"], as_index=False, dropna=False)["price"]
        .sum()
        .rename(columns={"price": "revenue"})
        .sort_values(["product_category_name_english", "month"], na_position="first")
    )


with timer("pandas: read + query") as q_pd:
    out_pd = pandas_query()
out_pd.tail()

pandas: read + query             0.68 s


,product_category_name_english,month,revenue
1238,watches_gifts,2018-04-01,88623.07
1239,watches_gifts,2018-05-01,119364.98
1240,watches_gifts,2018-06-01,85027.57
1241,watches_gifts,2018-07-01,95164.79
1242,watches_gifts,2018-08-01,69767.06


In [29]:
def polars_query():
    orders = pl.read_csv(DATA / "olist_orders_dataset.csv", try_parse_dates=True)
    items = pl.read_csv(DATA / "olist_order_items_dataset.csv")
    products = pl.read_csv(DATA / "olist_products_dataset.csv")
    trans = pl.read_csv(DATA / "product_category_name_translation.csv")

    delivered = orders.filter(pl.col("order_status") == "delivered").select("order_id", "order_purchase_timestamp")
    return (
        items.join(delivered, on="order_id")
        .join(products.select("product_id", "product_category_name"), on="product_id", how="left")
        .join(trans, on="product_category_name", how="left")
        .group_by(
            "product_category_name_english",
            pl.col("order_purchase_timestamp").dt.truncate("1mo").alias("month"),
        )
        .agg(revenue=pl.col("price").sum())
        .sort("product_category_name_english", "month", nulls_last=False)
    )


with timer("polars eager: read + query") as q_pl:
    out_pl = polars_query()
print(f"speed-up: {q_pd.seconds / q_pl.seconds:.1f}x")
out_pl.tail()

polars eager: read + query       0.16 s
speed-up: 4.1x


product_category_name_english,month,revenue
str,datetime[μs],f64
"""watches_gifts""",2018-04-01 00:00:00,88623.07
"""watches_gifts""",2018-05-01 00:00:00,119364.98
"""watches_gifts""",2018-06-01 00:00:00,85027.57
"""watches_gifts""",2018-07-01 00:00:00,95164.79
"""watches_gifts""",2018-08-01 00:00:00,69767.06


In [30]:
# Trust, then verify: same number of groups, same total revenue
print("groups  pandas:", len(out_pd), "| polars:", out_pl.height)
print("revenue pandas:", round(out_pd["revenue"].sum(), 2), "| polars:", round(out_pl["revenue"].sum(), 2))

groups  pandas: 1264 | polars: 1264
revenue pandas: 13221498.11 | polars: 13221498.11


## 3. Same query, lazy

Swap `read_csv` → `scan_csv` and add `.collect()` at the end. Polars now sees the **whole query** before running it.
In Polars 2.x, `collect()` on a lazy query runs on the **streaming engine** by default.

In [31]:
def polars_lazy_query():
    orders = pl.scan_csv(DATA / "olist_orders_dataset.csv", try_parse_dates=True)
    items = pl.scan_csv(DATA / "olist_order_items_dataset.csv")
    products = pl.scan_csv(DATA / "olist_products_dataset.csv")
    trans = pl.scan_csv(DATA / "product_category_name_translation.csv")

    delivered = orders.filter(pl.col("order_status") == "delivered").select("order_id", "order_purchase_timestamp")
    return (
        items.join(delivered, on="order_id")
        .join(products.select("product_id", "product_category_name"), on="product_id", how="left")
        .join(trans, on="product_category_name", how="left")
        .group_by(
            "product_category_name_english",
            pl.col("order_purchase_timestamp").dt.truncate("1mo").alias("month"),
        )
        .agg(revenue=pl.col("price").sum())
        .sort("product_category_name_english", "month", nulls_last=False)
    )


with timer("polars lazy: scan + query") as q_lazy:
    out_lazy = polars_lazy_query().collect()
print(f"speed-up vs pandas: {q_pd.seconds / q_lazy.seconds:.1f}x")

polars lazy: scan + query        0.08 s
speed-up vs pandas: 8.7x


In [32]:
# The optimized plan: look at the column lists under each CSV SCAN (projection pushdown)
print(polars_lazy_query().explain())

SORT BY [col("product_category_name_english"), col("month")]
  AGGREGATE[maintain_order: false]
    [col("price").sum().alias("revenue")] BY [col("product_category_name_english"), col("order_purchase_timestamp").dt.truncate(["1mo"]).alias("month")]
    FROM
    simple π 3/3 ["product_category_name_english", ... 2 other columns]
      LEFT JOIN:
      LEFT PLAN ON: [col("product_category_name")]
        simple π 3/3 ["price", ... 2 other columns]
          LEFT JOIN:
          LEFT PLAN ON: [col("product_id")]
            simple π 3/3 ["price", ... 2 other columns]
              INNER JOIN:
              LEFT PLAN ON: [col("order_id")]
                simple π 3/3 ["product_id", "price", ... 1 other column]
                  Csv SCAN [data/raw/olist_order_items_dataset.csv]
                  PROJECT 3/7 COLUMNS
                  ESTIMATED ROWS: 111070
              RIGHT PLAN ON: [col("order_id")]
                simple π 2/2 ["order_id", ... 1 other column]
                  Csv SCAN [

## Try it yourself

1. Restart the kernel, run `import os; os.environ["POLARS_MAX_THREADS"] = "1"` **before** `import polars`, and re-time the query. How much of the speed-up was multi-threading?
2. Which columns does the lazy plan read from `olist_orders_dataset.csv`? How many does the file have?